# Xuất phần cứng cho model chốt F2 (seed 1)
1. Nạp model F2 seed 1 + cấu hình lượng tử từ `improve_v4`.
2. **Mô phỏng số nguyên bit-exact** đúng như mạch HLS (AP_RND, AP_SAT, logit 14 bit thập phân) trên **toàn bộ DS2** → đây là **độ chính xác thật của phần cứng**, báo cáo trong bài.
3. Chọn tập con phân tầng (tối đa 250 nhịp mỗi lớp) làm vector test cho Vitis HLS.
4. **Sinh toàn bộ project HLS**: `ecg_f2.h` (kiểu dữ liệu theo frac_bits thật), `weights_f2.h`, `ecg_f2.cpp`, `ecg_f2_tb.cpp` + file test → ZIP tải về.

Chạy: **Runtime → Run all** (không cần GPU).

## Chuẩn bị (định nghĩa dùng chung)

In [2]:
import os, shutil
!pip install -q wfdb
from google.colab import drive
MP = '/content/drive'
if os.path.ismount(MP):
    print('Drive đã kết nối sẵn.')
else:
    if os.path.isdir(MP) and os.listdir(MP):
        shutil.rmtree(MP)          # thư mục tạm bị tạo nhầm, chưa phải Drive thật
    drive.mount(MP)

Mounted at /content/drive


## 1. Cấu hình chung

In [3]:
import os, json, time
import numpy as np

DATA_DIR = '/content/drive/MyDrive/mitdb_extracted/mit-bih-arrhythmia-database-1.0.0'   # thư mục chứa 100.dat, 100.hea, 100.atr ...
V4_DIR   = '/content/drive/MyDrive/ecg_processed/improve_v4'   # model F2 đã train
# OUT_DIR  = '/content/drive/MyDrive/ecg_processed/hw_f2'
# SEED_HW  = 1          # seed có jκ gần trung bình 5 seed nhất                 # nơi lưu kết quả, model
OUT_DIR  = '/content/drive/MyDrive/ecg_processed/hw_f2_s3'   # thư mục mới, không ghi đè bản seed 1
SEED_HW  = 3          # seed chọn theo CV trên DS1
os.makedirs(OUT_DIR, exist_ok=True)

FS_RAW = 360
SEED   = 42

# 4 lớp theo khuyến nghị AAMI EC57 / de Chazal (lớp Q gần như không có trong DS1/DS2
# vì các bản ghi máy tạo nhịp 102, 104, 107, 217 bị loại). Đặt USE_Q = True để giữ 5 lớp.
USE_Q = False

DS1 = [101, 106, 108, 109, 112, 114, 115, 116, 118, 119, 122, 124, 201, 203,
       205, 207, 208, 209, 215, 220, 223, 230]
DS2 = [100, 103, 105, 111, 113, 117, 121, 123, 200, 202, 210, 212, 213, 214,
       219, 221, 222, 228, 231, 232, 233, 234]

AAMI = {'N': 'N', 'L': 'N', 'R': 'N', 'e': 'N', 'j': 'N',
        'A': 'S', 'a': 'S', 'J': 'S', 'S': 'S',
        'V': 'V', 'E': 'V',
        'F': 'F',
        '/': 'Q', 'f': 'Q', 'Q': 'Q'}
BEAT_SYMBOLS = set(AAMI.keys())
CLASS_NAMES = ['N', 'S', 'V', 'F', 'Q'] if USE_Q else ['N', 'S', 'V', 'F']
NUM_CLASSES = len(CLASS_NAMES)
print('Số lớp:', NUM_CLASSES, CLASS_NAMES)

Số lớp: 4 ['N', 'S', 'V', 'F']


## 2. Tiền xử lý: lọc nhiễu, cắt nhịp, đặc trưng RR
- **Lọc**: loại trôi đường nền bằng 2 bộ lọc trung vị (200 ms, 600 ms) + thông thấp Butterworth 40 Hz (chạy trên CPU/PS, không tốn FPGA).
- **Cửa sổ**: tính theo giây nên đổi tần số lấy mẫu không làm lệch vị trí đỉnh R.
- **Chuẩn hóa từng nhịp**: trừ trung vị, chia giá trị tuyệt đối lớn nhất → khoảng [-1, 1] (khớp đầu vào 8-bit frac 6).
- **RR (10 số)**: RR trước, RR sau, RR trung bình 10 nhịp trước (nhân quả), 5 tỉ số so với nền 10/30 nhịp, 2 tỉ số so với **trung vị 300 nhịp** (chuẩn hóa theo bệnh nhân). Model chọn số đặc trưng dùng qua tham số `n_rr` (8 hoặc 10).

In [4]:
import wfdb
from scipy.ndimage import median_filter
from scipy.signal import butter, filtfilt

def load_record(rec):
    path = os.path.join(DATA_DIR, str(rec))
    r = wfdb.rdrecord(path)
    ch = r.sig_name.index('MLII') if 'MLII' in r.sig_name else 0
    sig = r.p_signal[:, ch].astype(np.float64)
    ann = wfdb.rdann(path, 'atr')
    return sig, np.asarray(ann.sample), np.asarray(ann.symbol)

def clean_signal(sig, fs=FS_RAW):
    base = median_filter(sig, size=int(0.2 * fs) | 1, mode='nearest')
    base = median_filter(base, size=int(0.6 * fs) | 1, mode='nearest')
    x = sig - base
    b, a = butter(4, 40.0 / (fs / 2), btype='low')
    return filtfilt(b, a, x)

def extract_record(rec, pre_s, post_s, ds, filt):
    sig, samples, symbols = load_record(rec)
    if filt:
        sig = clean_signal(sig)
    fs = FS_RAW / ds
    if ds > 1:
        sig = sig[::ds]                        # đã lọc thông thấp 40 Hz nên hạ mẫu an toàn
    pre, post = int(round(pre_s * fs)), int(round(post_s * fs))

    is_beat = np.array([s in BEAT_SYMBOLS for s in symbols])
    r_all = samples[is_beat] / ds
    sym_all = symbols[is_beat]
    rr = np.diff(r_all) / fs                   # giây

    X, R, Y = [], [], []
    for i in range(1, len(r_all) - 1):
        lab = AAMI[sym_all[i]]
        if lab == 'Q' and not USE_Q:
            continue
        c = int(round(r_all[i]))
        if c - pre < 0 or c + post > len(sig):
            continue
        w = sig[c - pre:c + post]
        w = w - np.median(w)
        m = np.max(np.abs(w))
        if m < 1e-9:
            continue
        pre_rr, post_rr = rr[i - 1], rr[i]
        local = np.mean(rr[max(0, i - 10):i])
        local30 = np.mean(rr[max(0, i - 30):i])
        med300 = np.median(rr[max(0, i - 300):i])
        X.append((w / m).astype(np.float32))
        R.append([pre_rr, post_rr, local, pre_rr / local, post_rr / local, pre_rr / post_rr,
                  pre_rr / local30, post_rr / local30, pre_rr / med300, post_rr / med300])
        Y.append(CLASS_NAMES.index(lab))
    return np.array(X), np.array(R, dtype=np.float32), np.array(Y), np.full(len(Y), rec)

_cache = {}
def build_dataset(records, pre_s, post_s, ds, filt):
    key = (tuple(records), pre_s, post_s, ds, filt)
    if key not in _cache:
        parts = [extract_record(r, pre_s, post_s, ds, filt) for r in records]
        _cache[key] = tuple(np.concatenate([p[k] for p in parts]) for k in range(4))
    return _cache[key]

def describe(y, name):
    cnt = np.bincount(y, minlength=NUM_CLASSES)
    print(name, dict(zip(CLASS_NAMES, cnt.tolist())), 'tổng', len(y))

## 3. Model thân thiện phần cứng
Cấu trúc giữ đúng các khối đã có IP HLS. Nhánh RR (nếu bật) ghép vào sau Flatten, trước Dense 1 — phần cứng chỉ thêm 6 thanh ghi đầu vào AXI-Lite. Đầu ra là **logit** (softmax bỏ đi vì phần cứng dùng argmax).

In [5]:
import tensorflow as tf
from tensorflow.keras import layers, models

N_RR = 10

def build_model(win_len, conv_blocks, use_rr, dense_units=32, rr_dense=0):
    beat_in = layers.Input(shape=(win_len, 1), name='beat')
    x = beat_in
    for i, (f, k) in enumerate(conv_blocks, 1):
        x = layers.Conv1D(f, k, padding='same', name=f'conv1d_{i}')(x)
        x = layers.ReLU(name=f'relu_{i}')(x)
        x = layers.MaxPooling1D(2, name=f'pool_{i}')(x)
    x = layers.Flatten(name='flatten')(x)
    inputs = [beat_in]
    if use_rr:
        rr_in = layers.Input(shape=(CUR_NRR,), name='rr')
        inputs.append(rr_in)
        r = layers.Dense(rr_dense, activation='relu', name='rr_dense')(rr_in) if rr_dense else rr_in
        x = layers.Concatenate(name='concat')([x, r])
    x = layers.Dense(dense_units, activation='relu', name='dense_1')(x)
    out = layers.Dense(NUM_CLASSES, name='dense_out')(x)   # logit
    return models.Model(inputs, out)

CUR_NRR = 10   # số đặc trưng RR dùng cho thí nghiệm hiện tại (đặt trong vòng lặp)

def rr_normalize(R):
    """Đưa RR về khoảng ~[0, 2]: giây chia 1.5, tỉ số chia 2. Cố định, không học -> dễ làm phần cứng."""
    R = R[:, :CUR_NRR]
    return (R / np.array([1.5, 1.5, 1.5, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0], dtype=np.float32)[:CUR_NRR]).clip(0, 3.9)

## 4. Huấn luyện: trọng số lớp, focal loss, tăng cường dữ liệu

In [6]:
def class_weights_effective(y, beta=0.999, max_ratio=10.0):
    cnt = np.maximum(np.bincount(y, minlength=NUM_CLASSES), 1)
    w = (1 - beta) / (1 - np.power(beta, cnt))
    w = w / w.min()
    w = np.minimum(w, max_ratio)
    return w / np.mean(w[np.bincount(y, minlength=NUM_CLASSES) > 0])

def focal_loss(alpha, gamma=2.0):
    alpha = tf.constant(alpha, dtype=tf.float32)
    def loss(y_true, logits):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        logp = tf.nn.log_softmax(logits)
        idx = tf.stack([tf.range(tf.shape(y_true)[0]), y_true], axis=1)
        lp = tf.gather_nd(logp, idx)
        p = tf.exp(lp)
        a = tf.gather(alpha, y_true)
        return tf.reduce_mean(-a * tf.pow(1.0 - p, gamma) * lp)
    return loss

def time_stretch(x, max_frac=0.10):
    """Co giãn cả batch theo 1 hệ số ngẫu nhiên, giữ tâm (đỉnh R) cố định."""
    L = tf.shape(x)[1]
    s = tf.random.uniform([], 1.0 - max_frac, 1.0 + max_frac)
    newL = tf.cast(tf.round(tf.cast(L, tf.float32) * s), tf.int32)
    y = tf.image.resize(x[:, :, None, :], [newL, 1], method='bilinear')[:, :, 0, :]
    diff = newL - L
    y = tf.cond(diff > 0,
                lambda: y[:, diff // 2: diff // 2 + L, :],
                lambda: tf.pad(y, [[0, 0], [(-diff) // 2, -diff - (-diff) // 2], [0, 0]], mode='SYMMETRIC'))
    return tf.reshape(y, tf.shape(x))

def augment_beats(x, stretch=False, stretch_frac=0.10):
    """x: (B, L, 1). Biên độ, nhiễu, trôi nền, dịch thời gian nhỏ (+ co giãn)."""
    if stretch:
        x = time_stretch(x, stretch_frac)
    B = tf.shape(x)[0]; L = tf.shape(x)[1]
    scale = tf.random.uniform([B, 1, 1], 0.85, 1.15)
    noise = tf.random.normal(tf.shape(x), stddev=0.02)
    t = tf.cast(tf.range(L), tf.float32)[None, :, None] / tf.cast(L, tf.float32)
    wander = tf.random.uniform([B, 1, 1], -0.08, 0.08) * tf.sin(
        2.0 * np.pi * tf.random.uniform([B, 1, 1], 0.2, 1.0) * t + tf.random.uniform([B, 1, 1], 0, 6.28))
    shift = tf.random.uniform([], -4, 5, dtype=tf.int32)
    x = tf.roll(x, shift, axis=1)
    return tf.clip_by_value(x * scale + noise + wander, -1.98, 1.98)

def make_ds(Xb, R, y, use_rr, batch, training, stretch=False, stretch_frac=0.10):
    feats = {'beat': Xb[..., None]}
    if use_rr:
        feats['rr'] = R
    ds = tf.data.Dataset.from_tensor_slices((feats, y))
    if training:
        ds = ds.shuffle(len(y), seed=SEED)
    ds = ds.batch(batch)
    if training:
        def aug(f, yy):
            f = dict(f); f['beat'] = augment_beats(f['beat'], stretch, stretch_frac); return f, yy
        ds = ds.map(aug, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

## 5. Đánh giá chuẩn AAMI + mô phỏng 8-bit + hiệu chỉnh bias

In [7]:
from sklearn.metrics import confusion_matrix

def aami_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    res = {}
    f1s, ses = [], []
    for c in range(NUM_CLASSES):
        tp = cm[c, c]; fn = cm[c].sum() - tp; fp = cm[:, c].sum() - tp
        se = tp / (tp + fn) if tp + fn else np.nan
        pp = tp / (tp + fp) if tp + fp else 0.0
        f1 = 2 * se * pp / (se + pp) if (se + pp) and not np.isnan(se) else 0.0
        res[CLASS_NAMES[c]] = dict(Se=se, PP=pp, F1=f1, n=int(cm[c].sum()))
        if cm[c].sum():
            f1s.append(f1); ses.append(se)
    res['acc'] = np.trace(cm) / cm.sum()
    res['macroF1'] = float(np.mean(f1s))
    res['balAcc'] = float(np.mean(ses))
    res['cm'] = cm
    return res

def print_metrics(m, title):
    print(f'--- {title} ---  acc={m["acc"]:.4f}  macroF1={m["macroF1"]:.4f}  balAcc={m["balAcc"]:.4f}')
    for c in CLASS_NAMES:
        d = m[c]
        if d['n']:
            print(f'   {c}: Se={d["Se"]:.3f}  +P={d["PP"]:.3f}  F1={d["F1"]:.3f}  (n={d["n"]})')
    print('   Confusion (hàng=thật, cột=đoán):'); print(m['cm'])

# ---------- Lượng tử hóa kiểu phần cứng (Q-format 8-bit, thang lũy thừa 2) ----------
def frac_bits_for(x, bits=8):
    m = float(np.max(np.abs(x)))
    if m < 1e-12:
        return bits - 1
    return int(max(bits - 1 - max(int(np.ceil(np.log2(m))), 0), 0))

def fq(x, f, bits=8):
    s = 2.0 ** f
    return np.clip(np.round(x * s), -2 ** (bits - 1), 2 ** (bits - 1) - 1) / s

def _conv_same(x, W, b):
    """x (n,L,C), W keras (k,C,F)."""
    k = W.shape[0]; pl = (k - 1) // 2; pr = k - 1 - pl
    xp = np.pad(x, ((0, 0), (pl, pr), (0, 0)))
    win = np.lib.stride_tricks.sliding_window_view(xp, k, axis=1)     # (n,L,C,k)
    return np.einsum('nlck,kcf->nlf', win, W, optimize=True) + b

def forward_np(model, Xb, R, conv_blocks, use_rr, qcfg=None, batch=2048, rr_dense=0):
    """Suy luận NumPy từng tầng; qcfg=None -> float, ngược lại fake-quant 8-bit như phần cứng."""
    Wd = {l.name: l.get_weights() for l in model.layers if l.get_weights()}
    Q = (lambda x, key: fq(x, qcfg[key])) if qcfg else (lambda x, key: x)
    outs = []
    for s in range(0, len(Xb), batch):
        x = Q(Xb[s:s + batch, :, None].astype(np.float64), 'input')
        for i in range(1, len(conv_blocks) + 1):
            W, b = Wd[f'conv1d_{i}']
            if qcfg:
                W, b = fq(W, qcfg[f'conv1d_{i}.kernel']), fq(b, qcfg[f'conv1d_{i}.bias'])
            x = Q(np.maximum(_conv_same(x, W, b), 0), f'relu_{i}')
            Lp = x.shape[1] // 2
            x = np.maximum(x[:, 0:2 * Lp:2], x[:, 1:2 * Lp:2])
        x = x.reshape(len(x), -1)
        if use_rr:
            r = Q(rr_normalize(R[s:s + batch]).astype(np.float64), 'rr')
            if rr_dense:
                W, b = Wd['rr_dense']
                if qcfg:
                    W, b = fq(W, qcfg['rr_dense.kernel']), fq(b, qcfg['rr_dense.bias'])
                r = Q(np.maximum(r @ W + b, 0), 'rr_dense')
            x = np.concatenate([x, r], axis=1)
        W, b = Wd['dense_1']
        if qcfg:
            W, b = fq(W, qcfg['dense_1.kernel']), fq(b, qcfg['dense_1.bias'])
        x = Q(np.maximum(x @ W + b, 0), 'dense_1')
        W, b = Wd['dense_out']
        if qcfg:
            W, b = fq(W, qcfg['dense_out.kernel']), fq(b, qcfg['dense_out.bias'])
        outs.append(x @ W + b)                    # logit giữ độ chính xác đầy đủ như phần cứng
    return np.concatenate(outs)

def calibrate(model, Xb, R, conv_blocks, use_rr, n=3000, rr_dense=0):
    """Chọn frac_bits cho trọng số + activation, hiệu chuẩn trên dữ liệu TRAIN."""
    idx = np.random.default_rng(SEED).choice(len(Xb), min(n, len(Xb)), replace=False)
    Wd = {l.name: l.get_weights() for l in model.layers if l.get_weights()}
    q = {'input': 6}
    for name, (W, b) in Wd.items():
        q[f'{name}.kernel'], q[f'{name}.bias'] = frac_bits_for(W), frac_bits_for(b)
    x = Xb[idx, :, None].astype(np.float64)
    for i in range(1, len(conv_blocks) + 1):
        W, b = Wd[f'conv1d_{i}']
        x = np.maximum(_conv_same(x, W, b), 0); q[f'relu_{i}'] = frac_bits_for(x)
        Lp = x.shape[1] // 2; x = np.maximum(x[:, 0:2 * Lp:2], x[:, 1:2 * Lp:2])
    x = x.reshape(len(x), -1)
    if use_rr:
        rr = rr_normalize(R[idx]); q['rr'] = frac_bits_for(rr)
        if rr_dense:
            W, b = Wd['rr_dense']; rr = np.maximum(rr @ W + b, 0); q['rr_dense'] = frac_bits_for(rr)
        x = np.concatenate([x, rr], axis=1)
    W, b = Wd['dense_1']; q['dense_1'] = frac_bits_for(np.maximum(x @ W + b, 0))
    return q

def tune_bias(logits, y, iters=3, grid=np.linspace(-3, 3, 61)):
    """Tìm độ lệch cộng vào logit từng lớp để tối đa macro-F1 trên VALIDATION."""
    off = np.zeros(NUM_CLASSES)
    best = aami_metrics(y, np.argmax(logits, 1))['macroF1']
    for _ in range(iters):
        present = set(np.unique(y).tolist())
        for c in [c for c in range(1, NUM_CLASSES) if c in present]:   # giữ N làm mốc; bỏ lớp vắng
            for g in grid:
                o = off.copy(); o[c] = g
                f = aami_metrics(y, np.argmax(logits + o, 1))['macroF1']
                if f > best + 1e-6:
                    best, off = f, o
    return off, best

## 6. Bộ chỉ số đầy đủ (AAMI + κ + jκ)
- **Sp** (specificity) = TN / (TN + FP) theo kiểu một-lớp-với-phần-còn-lại.
- **κ** (Cohen's kappa) = (p_o − p_e) / (1 − p_e).
- **jκ** = j/8 + κ/2, với j = Se_S + Se_V + +P_S + +P_V (thước đo chính trong nhiều bài inter-patient gần đây, nằm trong [0, 1]).

In [8]:
import pandas as pd, glob
pd.set_option('display.float_format', '{:.4f}'.format); pd.set_option('display.width', 250)

def full_metrics(y, p, names):
    k = len(names)
    cm = confusion_matrix(y, p, labels=list(range(k))).astype(np.int64)
    N = cm.sum()
    out = {}
    f1s, ses = [], []
    for c in range(k):
        tp = cm[c, c]; fn = cm[c].sum() - tp; fp = cm[:, c].sum() - tp; tn = N - tp - fn - fp
        se = tp / (tp + fn) if tp + fn else np.nan
        pp = tp / (tp + fp) if tp + fp else 0.0
        sp = tn / (tn + fp) if tn + fp else np.nan
        f1 = 2 * se * pp / (se + pp) if (tp + fn) and (se + pp) else 0.0
        out.update({f'Se_{names[c]}': se, f'PP_{names[c]}': pp, f'Sp_{names[c]}': sp, f'F1_{names[c]}': f1})
        if tp + fn:
            f1s.append(f1); ses.append(se)
    po = np.trace(cm) / N
    pe = float(np.sum(cm.sum(0) * cm.sum(1))) / N ** 2
    kappa = (po - pe) / (1 - pe)
    out.update(acc=po, macroF1=float(np.mean(f1s)), balAcc=float(np.mean(ses)), kappa=kappa)
    if 'S' in names and 'V' in names:
        j = out['Se_S'] + out['Se_V'] + out['PP_S'] + out['PP_V']
        out.update(j=j, jk=j / 8 + kappa / 2)
    return out, cm

def merge_F_into_V(a):
    a = a.copy(); a[a == 3] = 2; return a

def summarize(rows, group):
    df = pd.DataFrame(rows)
    num = [c for c in df.select_dtypes('number').columns if c not in group + ['seed']]
    pretty = df.groupby(group)[num].agg(lambda v: f'{np.nanmean(v):.4f} ± {np.nanstd(v, ddof=1) if len(v) > 1 else 0:.4f}')
    return df, pretty

def evaluate_saved(name_glob, out_tag):
    """Nạp các model đã lưu, trả về bảng chỉ số theo seed và confusion matrix tổng."""
    global CUR_NRR
    rows, rows_m, cms, per_rec = [], [], {}, []
    for jf in sorted(glob.glob(name_glob)):
        info = json.load(open(jf))
        E = info['cfg']; E['conv'] = [tuple(c) for c in E['conv']]
        CUR_NRR = max(E.get('n_rr', 0), 1)
        model = tf.keras.models.load_model(jf.replace('_quant.json', '.keras'), compile=False)
        Xte, Rte, yte, gte = build_dataset(DS2, E['pre_s'], E['post_s'], E['ds'], E['filt'])
        kw = dict(conv_blocks=E['conv'], use_rr=E['use_rr'], rr_dense=E['rr_dense'])
        lg_f = forward_np(model, Xte, Rte, **kw)
        lg_q = forward_np(model, Xte, Rte, qcfg=info['qcfg'], **kw)
        seed = info.get('seed', 0)
        for mode, lg in [('float', lg_f), ('int8', lg_q), ('int8+bias', lg_q + np.array(info['offset']))]:
            pred = lg.argmax(1)
            m, cm = full_metrics(yte, pred, CLASS_NAMES)
            rows.append(dict(exp=E['name'], mode=mode, seed=seed, **m))
            cms[(E['name'], mode)] = cms.get((E['name'], mode), 0) + cm
            mm, _ = full_metrics(merge_F_into_V(yte), merge_F_into_V(pred), ['N', 'S', 'V'])
            rows_m.append(dict(exp=E['name'], mode=mode, seed=seed, **mm))
            if mode == 'int8+bias':
                for rec in DS2:
                    mk = gte == rec; r = {'exp': E['name'], 'seed': seed, 'rec': rec}
                    for ci, c in enumerate(CLASS_NAMES):
                        sel = mk & (yte == ci)
                        r[f'n_{c}'] = int(sel.sum()); r[f'Se_{c}'] = (pred[sel] == ci).mean() if sel.sum() else np.nan
                    per_rec.append(r)
        print('Đã đánh giá', os.path.basename(jf))
    return rows, rows_m, cms, per_rec

## 7. Thư viện xuất phần cứng (lượng tử, mô phỏng bit-exact, sinh mã HLS)

In [9]:
# ============================================================
# Xuất phần cứng cho model F2: lượng tử hóa, mô phỏng số nguyên
# bit-exact (AP_RND/AP_SAT như Vitis HLS), sinh mã HLS
# ============================================================
import numpy as np, os

LF = 14          # số bit thập phân của logit trong phần cứng (ap_fixed<32,18>)
CLASS4 = ['N', 'S', 'V', 'F']

def q8(x, f):
    return np.clip(np.round(np.asarray(x, np.float64) * 2.0 ** f), -128, 127).astype(np.int64)

def hw_params(model, qcfg, offset, conv_blocks):
    """Trọng số nguyên + frac_bits theo đúng thứ tự chỉ số HLS."""
    W = {l.name: l.get_weights() for l in model.layers if l.get_weights()}
    P = {'conv': [], 'q': dict(qcfg)}
    fin = qcfg['input']
    for i, (F, K) in enumerate(conv_blocks, 1):
        k, b = W[f'conv1d_{i}']                       # keras (K, Cin, F)
        fw, fb, fa = qcfg[f'conv1d_{i}.kernel'], qcfg[f'conv1d_{i}.bias'], qcfg[f'relu_{i}']
        P['conv'].append(dict(W=q8(k, fw).transpose(2, 0, 1), B=q8(b, fb), fin=fin, fw=fw, fb=fb, fa=fa,
                              F=F, K=K, Cin=k.shape[1]))
        fin = fa
    k, b = W['rr_dense']                               # (8,16)
    P['rrd'] = dict(W=q8(k, qcfg['rr_dense.kernel']).T, B=q8(b, qcfg['rr_dense.bias']),
                    fin=qcfg['rr'], fw=qcfg['rr_dense.kernel'], fb=qcfg['rr_dense.bias'], fa=qcfg['rr_dense'])
    k, b = W['dense_1']                                # (1440+16, 32)
    P['d1'] = dict(W=q8(k, qcfg['dense_1.kernel']).T, B=q8(b, qcfg['dense_1.bias']),
                   fw=qcfg['dense_1.kernel'], fb=qcfg['dense_1.bias'], fa=qcfg['dense_1'],
                   fin_cnn=fin, fin_rr=qcfg['rr_dense'])
    k, b = W['dense_out']                              # (32, 4)
    fw, fb = qcfg['dense_out.kernel'], qcfg['dense_out.bias']
    Wo = q8(k, fw).T
    Bq = q8(b, fb)
    # bias + độ lệch hiệu chỉnh, gộp thành 1 hằng số ở đơn vị 2^-LF
    Btot = np.round(Bq * 2.0 ** (LF - fb) + np.asarray(offset) * 2.0 ** LF).astype(np.int64)
    P['do'] = dict(W=Wo, Btot=Btot, fw=fw, fb=fb, fin=P['d1']['fa'])
    return P

def _requant_relu(acc, units, fa):
    """acc (số nguyên, đơn vị 2^-units) -> ReLU -> ap_fixed 8-bit frac fa, AP_RND + AP_SAT."""
    s = units - fa
    if s > 0:
        v = (acc + (1 << (s - 1))) >> s
    else:
        v = acc << (-s)
    return np.where(acc > 0, np.clip(v, -128, 127), 0)

def _lin(x, W, B, fin, fw, fb):
    """x (…,Cin) nguyên, W (Cout,Cin), trả acc ở đơn vị 2^-U với U = max(fin+fw, fb)."""
    U = max(fin + fw, fb)
    return (x @ W.T) * (1 << (U - fin - fw)) + B * (1 << (U - fb)), U

def quant_inputs(Xb, R, rr_normalize, fin_rr):
    return q8(Xb, 6), q8(rr_normalize(R), fin_rr)

def hw_sim(P, xq, rq, batch=1024):
    """xq (n,L) int8 frac6, rq (n,8) int8 frac rr. Trả logits (đơn vị 2^-LF) và nhãn."""
    outs = []
    for s in range(0, len(xq), batch):
        x = xq[s:s + batch, :, None].astype(np.int64)
        for c in P['conv']:
            K = c['K']; pl = (K - 1) // 2; pr = K - 1 - pl
            xp = np.pad(x, ((0, 0), (pl, pr), (0, 0)))
            win = np.lib.stride_tricks.sliding_window_view(xp, K, axis=1)       # (n,L,Cin,K)
            U = max(c['fin'] + c['fw'], c['fb'])
            acc = np.einsum('nlck,fkc->nlf', win, c['W'], optimize=True) * (1 << (U - c['fin'] - c['fw'])) \
                  + c['B'] * (1 << (U - c['fb']))
            a = _requant_relu(acc, U, c['fa'])
            Lp = a.shape[1] // 2
            x = np.maximum(a[:, 0:2 * Lp:2], a[:, 1:2 * Lp:2])
        flat = x.reshape(len(x), -1)
        r = rq[s:s + batch].astype(np.int64)
        acc, U = _lin(r, P['rrd']['W'], P['rrd']['B'], P['rrd']['fin'], P['rrd']['fw'], P['rrd']['fb'])
        r16 = _requant_relu(acc, U, P['rrd']['fa'])
        d = P['d1']; nf = flat.shape[1]
        U = max(d['fin_cnn'] + d['fw'], d['fin_rr'] + d['fw'], d['fb'])
        acc = (flat @ d['W'][:, :nf].T) * (1 << (U - d['fin_cnn'] - d['fw'])) \
            + (r16 @ d['W'][:, nf:].T) * (1 << (U - d['fin_rr'] - d['fw'])) + d['B'] * (1 << (U - d['fb']))
        h = _requant_relu(acc, U, d['fa'])
        o = P['do']
        logit = (h @ o['W'].T) * (1 << (LF - o['fin'] - o['fw'])) + o['Btot']
        outs.append(logit)
    lg = np.concatenate(outs)
    return lg, lg.argmax(1)

# ---------------- Sinh mã HLS ----------------
def _arr(a):
    if a.ndim == 1:
        return '{' + ', '.join(str(int(v)) for v in a) + '}'
    return '{' + ',\n'.join(_arr(x) for x in a) + '}'

def _t(f, name, sat=False):
    return f'typedef ap_fixed<8, {8 - f}{", AP_RND, AP_SAT" if sat else ""}> {name};   // frac {f}'

def write_hls(P, L, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    c1, c2, c3 = P['conv']; rrd, d1, do = P['rrd'], P['d1'], P['do']
    NF = (L // 8) * c3['F']
    h = f"""// ============================================================
// ecg_f2.h - TU DONG SINH tu notebook ecg_f2_hw_export (model F2)
// Input: {L} mau ECG @180 Hz + 8 dac trung RR
// Conv({c1['F']},k{c1['K']})-Pool -> Conv({c2['F']},k{c2['K']})-Pool -> Conv({c3['F']},k{c3['K']})-Pool
// -> Flatten({NF}) ++ RR Dense(16) -> Dense(32) -> Dense(4) -> argmax
// ============================================================
#ifndef ECG_F2_H
#define ECG_F2_H
#include "ap_fixed.h"
#include "ap_int.h"

#define IN_LEN   {L}
#define N_RR     8
#define C1_F {c1['F']}
#define C1_K {c1['K']}
#define C2_F {c2['F']}
#define C2_K {c2['K']}
#define C3_F {c3['F']}
#define C3_K {c3['K']}
#define L1 IN_LEN
#define L2 (IN_LEN / 2)
#define L3 (IN_LEN / 4)
#define L4 (IN_LEN / 8)
#define N_FLAT   {NF}
#define RR_H     16
#define D1_IN    (N_FLAT + RR_H)
#define D1_OUT   32
#define NUM_CLASSES 4

{_t(6, 'in_t')}
{_t(P['q']['rr'], 'rr_t')}
{_t(c1['fw'], 'w1_t')}
{_t(c1['fb'], 'b1_t')}
{_t(c1['fa'], 'a1_t', True)}
{_t(c2['fw'], 'w2_t')}
{_t(c2['fb'], 'b2_t')}
{_t(c2['fa'], 'a2_t', True)}
{_t(c3['fw'], 'w3_t')}
{_t(c3['fb'], 'b3_t')}
{_t(c3['fa'], 'a3_t', True)}
{_t(rrd['fw'], 'wr_t')}
{_t(rrd['fb'], 'br_t')}
{_t(rrd['fa'], 'ar_t', True)}
{_t(d1['fw'], 'wd1_t')}
{_t(d1['fb'], 'bd1_t')}
{_t(d1['fa'], 'ad1_t', True)}
{_t(do['fw'], 'wo_t')}
typedef ap_fixed<48, 30>  acc_t;        // bo cong don (18 bit thap phan, du rong cho 1456 phep cong)
typedef ap_fixed<32, 18>  logit_t;      // logit, {LF} bit thap phan
typedef ap_uint<2>        label_t;      // 0=N 1=S 2=V 3=F

void ecg_f2_top(in_t ecg_in[IN_LEN], rr_t rr_in[N_RR], logit_t logits[NUM_CLASSES], label_t &label);
#endif
"""
    def lit(a, f):
        if a.ndim == 1:
            return '{' + ', '.join(repr(float(v) / 2 ** f) for v in a) + '}'
        return '{' + ',\n'.join(lit(x, f) for x in a) + '}'
    wts = ['// weights_f2.h - TU DONG SINH tu notebook, KHONG sua tay\n',
           '#ifndef WEIGHTS_F2_H\n#define WEIGHTS_F2_H\n#include "ecg_f2.h"\n\n']
    for nm, typ, arr, f, dims in [
            ('C1_W', 'w1_t', c1['W'], c1['fw'], '[C1_F][C1_K][1]'), ('C1_B', 'b1_t', c1['B'], c1['fb'], '[C1_F]'),
            ('C2_W', 'w2_t', c2['W'], c2['fw'], '[C2_F][C2_K][C1_F]'), ('C2_B', 'b2_t', c2['B'], c2['fb'], '[C2_F]'),
            ('C3_W', 'w3_t', c3['W'], c3['fw'], '[C3_F][C3_K][C2_F]'), ('C3_B', 'b3_t', c3['B'], c3['fb'], '[C3_F]'),
            ('RR_W', 'wr_t', rrd['W'], rrd['fw'], '[RR_H][N_RR]'), ('RR_B', 'br_t', rrd['B'], rrd['fb'], '[RR_H]'),
            ('D1_W', 'wd1_t', d1['W'], d1['fw'], '[D1_OUT][D1_IN]'), ('D1_B', 'bd1_t', d1['B'], d1['fb'], '[D1_OUT]'),
            ('DO_W', 'wo_t', do['W'], do['fw'], '[NUM_CLASSES][D1_OUT]')]:
        wts.append(f'static const {typ} {nm}{dims} =\n{lit(arr, f)};\n\n')
    wts.append(f'// bias dense_out + do lech hieu chinh (don vi 2^-{LF})\n')
    wts.append(f'static const logit_t DO_BTOT[NUM_CLASSES] = {lit(do["Btot"], LF)};\n\n#endif\n')

    def conv(n, IT, OT, L_, CIN, COUT, K, W_, B_):
        return f"""
// ---------- Conv1D({COUT}, k{K}, same) + ReLU: cua so truot ----------
static void conv{n}_relu({IT} in[{L_}][{CIN}], {OT} out[{L_}][{COUT}]) {{
#pragma HLS INLINE off
#pragma HLS ARRAY_PARTITION variable={W_} type=complete dim=2
#pragma HLS ARRAY_PARTITION variable={W_} type=complete dim=3
    {IT} win[{K}][{CIN}];
#pragma HLS ARRAY_PARTITION variable=win type=complete dim=0
    for (int k = 0; k < {K}; k++)
        for (int c = 0; c < {CIN}; c++)
            win[k][c] = 0;
    C{n}_STEP:
    for (int i = 0; i < {L_} + {K} - 1; i++) {{
        int new_pos = i - {(K - 1) // 2};
        int pos     = i - {K - 1};
        C{n}_SHIFT:
        for (int c = 0; c < {CIN}; c++) {{
#pragma HLS PIPELINE II=1
            for (int k = 0; k < {K - 1}; k++) win[k][c] = win[k + 1][c];
            win[{K - 1}][c] = (new_pos >= 0 && new_pos < {L_}) ? in[new_pos][c] : ({IT})0;
        }}
        if (pos >= 0 && pos < {L_}) {{
            C{n}_OC:
            for (int o = 0; o < {COUT}; o++) {{
#pragma HLS PIPELINE II=1
                acc_t acc = 0;
                for (int k = 0; k < {K}; k++)
                    for (int c = 0; c < {CIN}; c++)
                        acc += win[k][c] * {W_}[o][k][c];
                acc += {B_}[o];
                out[pos][o] = (acc > 0) ? ({OT})acc : ({OT})0;
            }}
        }}
    }}
}}
"""
    def pool(n, T, L_, C):
        return f"""
// ---------- MaxPool(2) ----------
static void pool{n}({T} in[{L_}][{C}], {T} out[{L_ // 2}][{C}]) {{
#pragma HLS INLINE off
    P{n}_POS:
    for (int p = 0; p < {L_ // 2}; p++) {{
        P{n}_CH:
        for (int c = 0; c < {C}; c++) {{
#pragma HLS PIPELINE II=1
            {T} a = in[2 * p][c], b = in[2 * p + 1][c];
            out[p][c] = (a > b) ? a : b;
        }}
    }}
}}
"""
    L1_, L2_, L3_, L4_ = L, L // 2, L // 4, L // 8
    cpp = f"""// ============================================================
// ecg_f2.cpp - TU DONG SINH tu notebook ecg_f2_hw_export (model F2)
// Giao tiep AXI-Lite: ecg_in[{L}], rr_in[8], logits[4], label
// ============================================================
#include "ecg_f2.h"
#include "weights_f2.h"
"""
    cpp += conv(1, 'in_t', 'a1_t', L1_, 1, c1['F'], c1['K'], 'C1_W', 'C1_B') + pool(1, 'a1_t', L1_, c1['F'])
    cpp += conv(2, 'a1_t', 'a2_t', L2_, c1['F'], c2['F'], c2['K'], 'C2_W', 'C2_B') + pool(2, 'a2_t', L2_, c2['F'])
    cpp += conv(3, 'a2_t', 'a3_t', L3_, c2['F'], c3['F'], c3['K'], 'C3_W', 'C3_B') + pool(3, 'a3_t', L3_, c3['F'])
    cpp += f"""
// ---------- Nhanh RR: Dense(8 -> 16) + ReLU ----------
static void rr_dense(rr_t in[N_RR], ar_t out[RR_H]) {{
#pragma HLS INLINE off
    RR_OUT:
    for (int o = 0; o < RR_H; o++) {{
        acc_t acc = RR_B[o];
        RR_IN:
        for (int i = 0; i < N_RR; i++) {{
#pragma HLS PIPELINE II=1
            acc += in[i] * RR_W[o][i];
        }}
        out[o] = (acc > 0) ? (ar_t)acc : (ar_t)0;
    }}
}}

// ---------- Flatten ++ RR -> Dense(32) + ReLU (tuan tu, 1 bo nhan) ----------
static void dense1_relu(a3_t in[{L4_}][{c3['F']}], ar_t rr[RR_H], ad1_t out[D1_OUT]) {{
#pragma HLS INLINE off
#pragma HLS BIND_STORAGE variable=D1_W type=rom_1p impl=bram
    D1_OUT_LOOP:
    for (int o = 0; o < D1_OUT; o++) {{
        acc_t acc = D1_B[o];
        D1_FLAT:
        for (int p = 0; p < {L4_}; p++) {{
            D1_CH:
            for (int c = 0; c < {c3['F']}; c++) {{
#pragma HLS PIPELINE II=1
                acc += in[p][c] * D1_W[o][p * {c3['F']} + c];
            }}
        }}
        D1_RR:
        for (int r = 0; r < RR_H; r++) {{
#pragma HLS PIPELINE II=1
            acc += rr[r] * D1_W[o][N_FLAT + r];
        }}
        out[o] = (acc > 0) ? (ad1_t)acc : (ad1_t)0;
    }}
}}

// ---------- Dense(4) + argmax ----------
static void dense_out_argmax(ad1_t in[D1_OUT], logit_t logits[NUM_CLASSES], label_t &label) {{
#pragma HLS INLINE off
    logit_t best = 0;
    label_t best_idx = 0;
    DO_OUT:
    for (int o = 0; o < NUM_CLASSES; o++) {{
        logit_t acc = DO_BTOT[o];
        DO_IN:
        for (int i = 0; i < D1_OUT; i++) {{
#pragma HLS PIPELINE II=1
            acc += in[i] * DO_W[o][i];
        }}
        logits[o] = acc;
        if (o == 0 || acc > best) {{ best = acc; best_idx = o; }}
    }}
    label = best_idx;
}}

// ---------- TOP ----------
void ecg_f2_top(in_t ecg_in[IN_LEN], rr_t rr_in[N_RR], logit_t logits[NUM_CLASSES], label_t &label) {{
#pragma HLS INTERFACE mode=s_axilite port=ecg_in
#pragma HLS INTERFACE mode=s_axilite port=rr_in
#pragma HLS INTERFACE mode=s_axilite port=logits
#pragma HLS INTERFACE mode=s_axilite port=label
#pragma HLS INTERFACE mode=s_axilite port=return
    in_t  x0[{L1_}][1];
    a1_t  r1[{L1_}][{c1['F']}];  a1_t p1[{L2_}][{c1['F']}];
    a2_t  r2[{L2_}][{c2['F']}];  a2_t p2[{L3_}][{c2['F']}];
    a3_t  r3[{L3_}][{c3['F']}];  a3_t p3[{L4_}][{c3['F']}];
    ar_t  rh[RR_H];
    ad1_t h[D1_OUT];
    for (int t = 0; t < IN_LEN; t++) {{
#pragma HLS PIPELINE II=1
        x0[t][0] = ecg_in[t];
    }}
    conv1_relu(x0, r1); pool1(r1, p1);
    conv2_relu(p1, r2); pool2(r2, p2);
    conv3_relu(p2, r3); pool3(r3, p3);
    rr_dense(rr_in, rh);
    dense1_relu(p3, rh, h);
    dense_out_argmax(h, logits, label);
}}
"""
    open(os.path.join(out_dir, 'ecg_f2.cpp'), 'w').write(cpp)
    open(os.path.join(out_dir, 'weights_f2.h'), 'w').write(''.join(wts))
    open(os.path.join(out_dir, 'ecg_f2.h'), 'w').write(h)

## 8. Nạp model F2 seed 1 và mô phỏng phần cứng trên toàn bộ DS2

In [10]:
info = json.load(open(os.path.join(V4_DIR, f'F2_rr8_str_s{SEED_HW}_quant.json')))
E = info['cfg']; E['conv'] = [tuple(c) for c in E['conv']]
CUR_NRR = E['n_rr']
model = tf.keras.models.load_model(os.path.join(V4_DIR, f'F2_rr8_str_s{SEED_HW}.keras'), compile=False)
model.summary()
qcfg, offset = info['qcfg'], np.array(info['offset'])
print('frac_bits:', qcfg); print('offset bias:', offset)

Xte, Rte, yte, gte = build_dataset(DS2, E['pre_s'], E['post_s'], E['ds'], E['filt'])
P = hw_params(model, qcfg, offset, E['conv'])
xq, rq = quant_inputs(Xte, Rte, rr_normalize, qcfg['rr'])
t0 = time.time()
lg_hw, pred_hw = hw_sim(P, xq, rq)
print(f'Mô phỏng {len(yte)} nhịp trong {time.time() - t0:.0f} s')

# Đối chiếu với kết quả fake-quant (int8+bias) đã báo cáo ở vòng 4-5
lg_fq = forward_np(model, Xte, Rte, E['conv'], True, qcfg=qcfg, rr_dense=E['rr_dense']) + offset
print('Tỉ lệ nhãn trùng giữa mô phỏng phần cứng và fake-quant:', (lg_fq.argmax(1) == pred_hw).mean())

m_hw, cm_hw = full_metrics(yte, pred_hw, CLASS_NAMES)
print('\n=== ĐỘ CHÍNH XÁC PHẦN CỨNG (bit-exact) trên toàn DS2 ===')
display(pd.Series(m_hw).to_frame('F2 seed 1 - phần cứng'))
display(pd.DataFrame(cm_hw, index=[f'thật {c}' for c in CLASS_NAMES], columns=[f'đoán {c}' for c in CLASS_NAMES]))

Model: "functional_13"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ beat (InputLayer)   │ (None, 360, 1)    │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 360, 16)   │        128 │ beat[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu_1 (ReLU)       │ (None, 360, 16)   │          0 │ conv1d_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_1              │ (None, 180, 16)   │          0 │ relu_1[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 180, 32)   │      2,592 │ pool_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu_2 (ReLU)       │ (None, 180, 32)   │          0 │ conv1d_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_2              │ (None, 90, 32)    │          0 │ relu_2[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 90, 32)    │      5,152 │ pool_2[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu_3 (ReLU)       │ (None, 90, 32)    │          0 │ conv1d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_3              │ (None, 45, 32)    │          0 │ relu_3[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rr (InputLayer)     │ (None, 8)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 1440)      │          0 │ pool_3[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rr_dense (Dense)    │ (None, 16)        │        144 │ rr[0][0]          │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat              │ (None, 1456)      │          0 │ flatten[0][0],    │
│ (Concatenate)       │                   │            │ rr_dense[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 32)        │     46,624 │ concat[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_out (Dense)   │ (None, 4)         │        132 │ dense_1[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 54,772 (213.95 KB)

 Trainable params: 54,772 (213.95 KB)

 Non-trainable params: 0 (0.00 B)

frac_bits: {'input': 6, 'conv1d_1.kernel': 7, 'conv1d_1.bias': 7, 'conv1d_2.kernel': 7, 'conv1d_2.bias': 7, 'conv1d_3.kernel': 7, 'conv1d_3.bias': 7, 'rr_dense.kernel': 7, 'rr_dense.bias': 7, 'dense_1.kernel': 7, 'dense_1.bias': 7, 'dense_out.kernel': 7, 'dense_out.bias': 7, 'relu_1': 6, 'relu_2': 5, 'relu_3': 4, 'rr': 5, 'rr_dense': 5, 'dense_1': 4}
offset bias: [ 0.          0.06666667 -0.53333333 -1.1       ]
Mô phỏng 49649 nhịp trong 254 s
Tỉ lệ nhãn trùng giữa mô phỏng phần cứng và fake-quant: 0.9984289713790812

=== ĐỘ CHÍNH XÁC PHẦN CỨNG (bit-exact) trên toàn DS2 ===


,F2 seed 1 - phần cứng
Se_N,0.9785
PP_N,0.9606
Sp_N,0.6744
F1_N,0.9695
Se_S,0.3110
PP_S,0.3531
Sp_S,0.9781
F1_S,0.3307
Se_V,0.8760
PP_V,0.9598


,đoán N,đoán S,đoán V,đoán F
thật N,43254,880,65,7
thật S,1244,571,21,0
thật V,204,164,2820,31
thật F,324,2,32,30


In [11]:
m = (yte == 1) & (gte != 232)
print('Nhip S ngoai ban ghi 232:', m.sum(), '| doan dung S:', int((pred_hw[m] == 1).sum()),
      '| Se =', round((pred_hw[m] == 1).mean(), 3))

Nhip S ngoai ban ghi 232: 455 | doan dung S: 274 | Se = 0.602


In [12]:
TB_SRC = r'''// ============================================================
// ecg_f2_tb.cpp - Testbench IP ecg_f2_top (model F2)
// File trong Test Bench (xuat tu notebook ecg_f2_hw_export):
//   test_beats_q.txt   : moi dong IN_LEN so nguyen int8 (nhip ECG, frac 6)
//   test_rr_q.txt      : moi dong 8 so nguyen int8 (dac trung RR)
//   test_labels_true.txt, test_labels_sim.txt
//   test_logits_sim.txt: 4 so nguyen / dong, don vi 2^-14
// Input Arguments:  (trong) = toan bo | 10 = 10 mau dau (Co-sim) | demo 37 = in chi tiet nhip 37
// PASS khi logit + nhan khop bit-exact 100% voi mo phong Python
// ============================================================
#include "ecg_f2.h"
#include <fstream>
#include <iostream>
#include <iomanip>
#include <cstdlib>
#include <cstring>

#define N_MAX 2000
static in_t ecg[N_MAX][IN_LEN];
static rr_t rr[N_MAX][N_RR];
static int  y_true[N_MAX], y_sim[N_MAX];
static long lg_sim[N_MAX][NUM_CLASSES];
static const char *CLS[NUM_CLASSES] = {"N", "S", "V", "F"};

template <typename T> T raw8(int v) { ap_int<8> r = v; T x; x.range(7, 0) = r.range(7, 0); return x; }

static int load() {
    std::ifstream fb("test_beats_q.txt"), fr("test_rr_q.txt"), ft("test_labels_true.txt"),
                  fs("test_labels_sim.txt"), fl("test_logits_sim.txt");
    if (!fb || !fr || !ft || !fs || !fl) { std::cerr << "Thieu file test_*.txt\n"; return -1; }
    int n = 0, v;
    while (n < N_MAX && (fb >> v)) {
        ecg[n][0] = raw8<in_t>(v);
        for (int t = 1; t < IN_LEN; t++) { fb >> v; ecg[n][t] = raw8<in_t>(v); }
        for (int i = 0; i < N_RR; i++) { fr >> v; rr[n][i] = raw8<rr_t>(v); }
        ft >> y_true[n]; fs >> y_sim[n];
        for (int o = 0; o < NUM_CLASSES; o++) fl >> lg_sim[n][o];
        n++;
    }
    return n;
}

int main(int argc, char *argv[]) {
    int n = load();
    if (n < 0) return 1;
    int only = -1;
    if (argc > 2 && strcmp(argv[1], "demo") == 0) only = atoi(argv[2]);
    else if (argc > 1 && atoi(argv[1]) > 0 && atoi(argv[1]) < n) n = atoi(argv[1]);
    std::cout << "So mau: " << (only >= 0 ? 1 : n) << "\n";

    int lerr = 0, lab_err = 0, ok = 0, cnt = 0;
    int conf[NUM_CLASSES][NUM_CLASSES] = {{0}};
    for (int s = (only >= 0 ? only : 0); s < (only >= 0 ? only + 1 : n); s++) {
        logit_t lg[NUM_CLASSES]; label_t lab;
        ecg_f2_top(ecg[s], rr[s], lg, lab);
        bool same = true;
        for (int o = 0; o < NUM_CLASSES; o++)
            if ((long)(lg[o] * 16384).to_int64() != lg_sim[s][o]) { same = false; }
        if (!same) lerr++;
        if ((int)lab != y_sim[s]) lab_err++;
        if ((int)lab == y_true[s]) ok++;
        conf[y_true[s]][(int)lab]++; cnt++;
        if (only >= 0) {
            std::cout << "Nhip #" << s << " | logit:";
            for (int o = 0; o < NUM_CLASSES; o++) std::cout << " " << CLS[o] << "=" << lg[o].to_double();
            std::cout << "\n  Phan cung: " << CLS[(int)lab] << " | That: " << CLS[y_true[s]]
                      << " | bit-exact: " << (same ? "CO" : "KHONG") << "\n";
        }
    }
    std::cout << std::fixed << std::setprecision(4);
    std::cout << "Logit lech: " << lerr << " | Nhan lech voi mo phong: " << lab_err
              << " | Accuracy (so nhan that): " << (double)ok / cnt << "\n";
    std::cout << "Confusion (hang=that, cot=du doan)\n      N     S     V     F\n";
    for (int r = 0; r < NUM_CLASSES; r++) {
        std::cout << "  " << CLS[r];
        for (int c = 0; c < NUM_CLASSES; c++) std::cout << std::setw(6) << conf[r][c];
        std::cout << "\n";
    }
    if (lerr == 0 && lab_err == 0) { std::cout << "PASS: khop bit-exact voi mo phong Python\n"; return 0; }
    std::cout << "FAIL\n"; return 1;
}
'''
print('Đã nạp mã testbench', len(TB_SRC), 'ký tự')

Đã nạp mã testbench 3684 ký tự


## 9. Vector test cho Vitis HLS và sinh project

In [13]:
HLS_DIR = '/content/ECG_F2_HLS'
if os.path.exists(HLS_DIR): shutil.rmtree(HLS_DIR)
write_hls(P, Xte.shape[1], HLS_DIR)
open(os.path.join(HLS_DIR, 'ecg_f2_tb.cpp'), 'w').write(TB_SRC)

rng = np.random.default_rng(SEED)
idx = np.concatenate([rng.choice(np.where(yte == c)[0], min(250, (yte == c).sum()), replace=False)
                      for c in range(NUM_CLASSES)])
rng.shuffle(idx)
for nm, arr in [('test_beats_q', xq[idx]), ('test_rr_q', rq[idx]), ('test_labels_true', yte[idx]),
                ('test_labels_sim', pred_hw[idx]), ('test_logits_sim', lg_hw[idx])]:
    np.savetxt(os.path.join(HLS_DIR, nm + '.txt'), arr, fmt='%d')
np.save(os.path.join(HLS_DIR, 'test_indices_in_DS2.npy'), idx)
print('Số vector test:', len(idx), dict(zip(CLASS_NAMES, np.bincount(yte[idx], minlength=4))))

json.dump(dict(qcfg=qcfg, offset=offset.tolist(), LF=LF, seed=SEED_HW, cfg=E,
               metrics_hw_DS2={k: float(v) for k, v in m_hw.items()}),
          open(os.path.join(HLS_DIR, 'f2_hw_config.json'), 'w'), indent=1)
os.makedirs(OUT_DIR, exist_ok=True)
shutil.make_archive(os.path.join(OUT_DIR, 'ECG_F2_HLS'), 'zip', HLS_DIR)
print(sorted(os.listdir(HLS_DIR)))
from google.colab import files
files.download(os.path.join(OUT_DIR, 'ECG_F2_HLS.zip'))

Số vector test: 1000 {'N': np.int64(250), 'S': np.int64(250), 'V': np.int64(250), 'F': np.int64(250)}
['ecg_f2.cpp', 'ecg_f2.h', 'ecg_f2_tb.cpp', 'f2_hw_config.json', 'test_beats_q.txt', 'test_indices_in_DS2.npy', 'test_labels_sim.txt', 'test_labels_true.txt', 'test_logits_sim.txt', 'test_rr_q.txt', 'weights_f2.h']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
# ============================================================
# Chay THEM cell nay o CUOI notebook ecg_f2_hw_export (Colab)
# -> tao f2_params.npz cho baseline CPU/ARM/GPU
# ============================================================
Wd = {l.name: l.get_weights() for l in model.layers if l.get_weights()}
F = dict(conv=[(Wd[f'conv1d_{i}'][0], Wd[f'conv1d_{i}'][1], Wd[f'conv1d_{i}'][0].shape[0])
               for i in range(1, len(E['conv']) + 1)],
         rrd=tuple(Wd['rr_dense']), d1=tuple(Wd['dense_1']), do=tuple(Wd['dense_out']),
         offset=offset)
np.savez_compressed(os.path.join(HLS_DIR, 'f2_params.npz'), P=P, F=F, qcfg=qcfg)

# Xuat them TOAN BO DS2 (49.649 nhip) de do accuracy + throughput tren board
np.savetxt(os.path.join(HLS_DIR, 'ds2_beats_q.txt'), xq, fmt='%d')
np.savetxt(os.path.join(HLS_DIR, 'ds2_rr_q.txt'), rq, fmt='%d')
np.savetxt(os.path.join(HLS_DIR, 'ds2_labels_true.txt'), yte, fmt='%d')
np.save(os.path.join(HLS_DIR, 'ds2_X_float.npy'), Xte.astype(np.float32))
np.save(os.path.join(HLS_DIR, 'ds2_RR_float.npy'), rr_normalize(Rte).astype(np.float32))
print('Da xuat f2_params.npz va bo test DS2 day du')

shutil.make_archive(os.path.join(OUT_DIR, 'ECG_F2_BOARD'), 'zip', HLS_DIR)
files.download(os.path.join(OUT_DIR, 'ECG_F2_BOARD.zip'))


Da xuat f2_params.npz va bo test DS2 day du


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>